# Segmentation alimentaire avec FoodSeg103 et YOLOv11-seg

## Objectif

Ce notebook entraîne un modèle de segmentation alimentaire à partir du dataset FoodSeg103.

L’objectif est de segmenter les zones alimentaires dans une image afin d’estimer la proportion occupée par les aliments dans un plat.

## Dataset

Source : FoodSeg103 sur Kaggle  
Chemin Kaggle : `/kaggle/input/datasets/fontainenathan/foodseg103`

Le dataset contient :
- des images dans `Images/img_dir`
- des masques de segmentation dans `Images/ann_dir`
- les listes d’entraînement et de test dans `ImageSets`

## Sorties

Le notebook génère :
- un dataset converti au format YOLO segmentation
- un modèle entraîné `best.pt`
- des résultats de validation

In [ ]:
# ==========================================================
# Configuration de la source des données
# ==========================================================

USE_DVC = True


# Limite souhaitée
LIMIT_TRAIN = 30

import os
import subprocess
import sys
import importlib.metadata

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip()

print("GPU détecté :", gpu_info)

torch_version = None
try:
    torch_version = importlib.metadata.version("torch")
except:
    pass

if "P100" in gpu_info:
    if torch_version != "2.6.0":
        print("Installation PyTorch compatible P100...")

        subprocess.run([
            sys.executable, "-m", "pip", "install",
            "torch==2.6.0",
            "torchvision==0.21.0",
            "torchaudio==2.6.0",
            "--index-url",
            "https://download.pytorch.org/whl/cu126"
        ])

    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "ultralytics"
    ])

else:
    print("GPU moderne détecté, installation légère.")

    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "ultralytics"
    ])

FileNotFoundError: [WinError 2] Le fichier spécifié est introuvable

In [ ]:
# Import des librairies nécessaires.
import os
import cv2
import shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from IPython.display import FileLink

In [ ]:
if USE_DVC:

    !pip install -q dvc dvc-azure

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    CONNECTION_STRING = secrets.get_secret(
        "AZURE_STORAGE_CONNECTION_STRING"
    )
    
    !dvc remote modify azure connection_string "$CONNECTION_STRING"

    !dvc pull

In [ ]:
# Chemin racine du dataset FoodSeg103.
if USE_DVC:
    ROOT = "/kaggle/working/dataset"
else:
    ROOT = "/kaggle/input/datasets/cleristonmartinelo/foodsegmentation"
    
# Dossiers des images et des masques.
IMG_DIR = f"{ROOT}/Images/img_dir"
ANN_DIR = f"{ROOT}/Images/ann_dir"

# Fichier des catégories.
CATEGORY_FILE = f"{ROOT}/category_id.txt"

# Listes train/test.
TRAIN_LIST = f"{ROOT}/ImageSets/train.txt"
TEST_LIST = f"{ROOT}/ImageSets/test.txt"

# Dossier de sortie converti au format YOLO.
DEST = "/kaggle/working/dataset"

print("ROOT:", ROOT)
print("IMG_DIR:", IMG_DIR)
print("ANN_DIR:", ANN_DIR)
print("CATEGORY_FILE:", CATEGORY_FILE)
print("TRAIN_LIST:", TRAIN_LIST)
print("TEST_LIST:", TEST_LIST)


In [ ]:
import random

# Lecture des fichiers train.txt et test.txt.
def lire_liste(path):
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

train_files = lire_liste(TRAIN_LIST)
val_files = lire_liste(TEST_LIST)

# Sauvegarde des tailles originales
original_train = len(train_files)
original_val = len(val_files)

# Calcul du ratio original val/train
val_ratio = original_val / original_train

# Nombre proportionnel de validation
if LIMIT_TRAIN is not None:
    LIMIT_VAL = int(LIMIT_TRAIN * val_ratio)
else:
    LIMIT_VAL = None

# Randomisation pour éviter le biais
import random

# Seed aléatoire à chaque exécution
SEED = random.randint(1, 999999)

print("Seed utilisée :", SEED)

random.shuffle(train_files)
random.shuffle(val_files)

# Réduction du dataset
if LIMIT_TRAIN is not None:
    train_files = train_files[:LIMIT_TRAIN]
    val_files = val_files[:LIMIT_VAL]

print("Train original:", original_train)
print("Val original:", original_val)
print("Nouveau train:", len(train_files))
print("Nouveau val:", len(val_files))
print("Ratio conservé:", round(len(val_files) / len(train_files), 4))

print("Exemples train:", train_files[:5])
print("Exemples val:", val_files[:5])

In [ ]:
# Lecture des catégories FoodSeg103.
# Le masque utilise : 0 = background, 1 à 103 = classes alimentaires.
# YOLO doit utiliser : 0 à 102 = classes alimentaires, sans background.

categories_foodseg = {}

with open(CATEGORY_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(None, 1)  # sépare l'ID et le nom, même si le nom contient des espaces
        if len(parts) == 2:
            id_foodseg = int(parts[0])
            nom_classe = parts[1].strip()
            categories_foodseg[id_foodseg] = nom_classe

# On ignore le background et on décale les IDs pour YOLO.
categories_yolo = {
    id_foodseg - 1: nom
    for id_foodseg, nom in categories_foodseg.items()
    if id_foodseg != 0
}

print("Nombre de classes YOLO:", len(categories_yolo))
print("Exemples:")
for i in list(categories_yolo.keys())[:10]:
    print(i, "->", categories_yolo[i])

# Exemple important : FoodSeg 66 = rice, donc YOLO 65 = rice.
print("FoodSeg 48 -> YOLO", 48 - 1, "=", categories_foodseg.get(48))
print("FoodSeg 66 -> YOLO", 66 - 1, "=", categories_foodseg.get(66))
print("FoodSeg 90 -> YOLO", 90 - 1, "=", categories_foodseg.get(90))


In [ ]:
# Création de la structure attendue par YOLO.
for d in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{DEST}/{d}", exist_ok=True)

print("Structure YOLO créée.")

In [ ]:
# Conversion d'un masque PNG en fichier label YOLO segmentation multi-classes.
# Important : on conserve les vraies catégories du masque FoodSeg103.
# 0 = background, ignoré.
# 1 à 103 = classes alimentaires FoodSeg103.
# YOLO attend des classes de 0 à 102, donc : classe_yolo = valeur_masque - 1.

def mask_to_yolo_multiclass(mask_path, output_txt, min_area=20):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        return False

    h, w = mask.shape
    lignes = []

    valeurs_classes = np.unique(mask)

    for valeur in valeurs_classes:
        if valeur == 0:
            continue  # background

        classe_yolo = int(valeur) - 1

        # Sécurité : ignorer une classe inconnue.
        if classe_yolo not in categories_yolo:
            continue

        masque_classe = (mask == valeur).astype(np.uint8) * 255

        contours, _ = cv2.findContours(
            masque_classe,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < 3:
                continue

            if cv2.contourArea(contour) < min_area:
                continue

            # Simplification légère pour éviter des polygones trop longs.
            epsilon = 0.001 * cv2.arcLength(contour, True)
            contour = cv2.approxPolyDP(contour, epsilon, True)

            if len(contour) < 3:
                continue

            points = contour.reshape(-1, 2)
            coords = []

            for x, y in points:
                coords.append(x / w)
                coords.append(y / h)

            ligne = str(classe_yolo) + " " + " ".join([f"{c:.6f}" for c in coords])
            lignes.append(ligne)

    with open(output_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(lignes))

    return len(lignes) > 0


In [ ]:
# Fonction pour trouver le masque correspondant à une image.
# Exemple : 00000001.jpg -> 00000001.png

def trouver_masque(nom_image, split):
    base = Path(nom_image).stem
    mask_path = f"{ANN_DIR}/{split}/{base}.png"

    if os.path.exists(mask_path):
        return mask_path

    return None


# Conversion d'un split train ou test vers la structure YOLO.
def convertir_split(files, src_split, dst_split):
    ok = 0
    manquants = 0
    sans_masque_valide = 0

    for nom_image in files:
        img_path = f"{IMG_DIR}/{src_split}/{nom_image}"
        mask_path = trouver_masque(nom_image, src_split)

        if not os.path.exists(img_path) or mask_path is None:
            manquants += 1
            continue

        base = Path(nom_image).stem

        dst_img = f"{DEST}/images/{dst_split}/{nom_image}"
        dst_label = f"{DEST}/labels/{dst_split}/{base}.txt"

        shutil.copy(img_path, dst_img)

        if mask_to_yolo_multiclass(mask_path, dst_label):
            ok += 1
        else:
            sans_masque_valide += 1

    print(f"{dst_split} ok:", ok)
    print(f"{dst_split} manquants:", manquants)
    print(f"{dst_split} sans masque valide:", sans_masque_valide)


convertir_split(train_files, "train", "train")
convertir_split(val_files, "test", "val")


In [ ]:
# Vérification finale de la conversion.
print("Images train:", len(os.listdir(f"{DEST}/images/train")))
print("Labels train:", len(os.listdir(f"{DEST}/labels/train")))
print("Images val:", len(os.listdir(f"{DEST}/images/val")))
print("Labels val:", len(os.listdir(f"{DEST}/labels/val")))

print("Exemples labels:", os.listdir(f"{DEST}/labels/train")[:5])

In [ ]:
# Vérification du contenu d'un fichier label.
exemple_label = os.listdir(f"{DEST}/labels/train")[0]

with open(f"{DEST}/labels/train/{exemple_label}", "r") as f:
    print(f.read()[:500])

In [ ]:
# Création du fichier de configuration YOLO multi-classes.
# On utilise les 103 classes alimentaires de FoodSeg103, sans le background.

yaml_content = f"""
path: {DEST}
train: images/train
val: images/val

names:
"""

for idx in sorted(categories_yolo.keys()):
    nom = categories_yolo[idx].replace("'", "")
    yaml_content += f"  {idx}: '{nom}'\n"

with open("/kaggle/working/dataset.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(yaml_content[:1500])


In [ ]:
# ==========================================================
# Configuration MLflow
# ==========================================================

!pip install mlflow

import mlflow
from kaggle_secrets import UserSecretsClient
from ultralytics import YOLO

user_secrets = UserSecretsClient()

#MLFLOW_USER = user_secrets.get_secret("MLFLOW_USER")
#MLFLOW_PASS = user_secrets.get_secret("MLFLOW_PASS")
#MLFLOW_HOST = user_secrets.get_secret("MLFLOW_HOST")

MLFLOW_USER = "mlflowsuperuser"
MLFLOW_PASS = "wfu:1H6B_pFLe',2\""
MLFLOW_HOST = "cleristonm.duckdns.org"

TRACKING_URI = f"http://{MLFLOW_USER}:{MLFLOW_PASS}@{MLFLOW_HOST}:5000"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("foodseg-yolo")

# ==========================================================
# Paramètres
# ==========================================================

MODEL_NAME = "yolo11s-seg.pt"
EPOCHS = 50
IMGSZ = 768
BATCH = 8

# ==========================================================
# Training + Logging
# ==========================================================

with mlflow.start_run() as run:

    # --------------------------
    # Paramètres
    # --------------------------
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("imgsz", IMGSZ)
    mlflow.log_param("batch", BATCH)
    mlflow.log_param("seed", SEED)
    mlflow.log_param("limit_train", LIMIT_TRAIN)

    # --------------------------
    # Tags
    # --------------------------
    mlflow.set_tag("task", "segmentation")
    mlflow.set_tag("dataset", "foodseg103")

    # --------------------------
    # Entraînement
    # --------------------------
    model = YOLO(MODEL_NAME)

    train_results = model.train(
        data="/kaggle/working/dataset.yaml",
        task="segment",
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED
    )

    # --------------------------
    # Validation
    # --------------------------
    metrics = model.val()

    # --------------------------
    # Métriques - Box
    # --------------------------
    mlflow.log_metric("map50_box", metrics.box.map50)
    mlflow.log_metric("map75_box", metrics.box.map75)
    mlflow.log_metric("map_box", metrics.box.map)

    # --------------------------
    # Métriques - Mask
    # --------------------------
    mlflow.log_metric("map50_mask", metrics.seg.map50)
    mlflow.log_metric("map75_mask", metrics.seg.map75)
    mlflow.log_metric("map_mask", metrics.seg.map)

    # Tags (visibles dans MLflow)
    mlflow.set_tag("map50_mask", f"{metrics.seg.map50:.4f}")
    mlflow.set_tag("map_mask", f"{metrics.seg.map:.4f}")
    mlflow.set_tag("map50_box", f"{metrics.box.map50:.4f}")

    # --------------------------
    # Artifacts
    # --------------------------
    run_dir = train_results.save_dir

    # Sauvegarde complète du dossier d'entraînement
    mlflow.log_artifacts(run_dir, artifact_path="training")

    # --------------------------
    # Enregistrement du modèle
    # --------------------------
    model_uri = f"runs:/{run.info.run_id}/training/weights/best.pt"

    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name="foodseg-yolo"
    )

    print("=" * 60)
    print(f"Run ID           : {run.info.run_id}")
    print(f"Model Version    : {registered_model.version}")
    print(f"mAP50 Mask       : {metrics.seg.map50:.4f}")
    print(f"mAP Mask         : {metrics.seg.map:.4f}")
    print("=" * 60)

In [ ]:
metrics = model.val()

new_map = metrics.box.map50 if hasattr(metrics, "box") else metrics.seg.map50

print("New model mAP50:", new_map)



In [ ]:
import json
import os

old_map = 0

if os.path.exists("/kaggle/input/current-model/metrics.json"):
    with open("/kaggle/input/current-model/metrics.json", "r") as f:
        old_map = json.load(f)["mAP50"]

print("Current production mAP50:", old_map)

In [ ]:
# Test sur des images de validation.
model.predict(
    source=f"{DEST}/images/val",
    conf=0.25,
    save=True,
    max_det=10
)

In [ ]:
# Trouver le modèle entraîné.
!find /kaggle/working -name "best.pt"

In [ ]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

if new_map > old_map:
    print("New model is better. Promoting.")

    !cp /kaggle/working/runs/segment/train*/weights/best.pt /kaggle/working/best.pt

    with open("/kaggle/working/metrics.json", "w") as f:
        json.dump({"mAP50": float(new_map)}, f)

else:
    print("New model is worse. Keeping current production model.")

# Lecture du fichier de résultats du training
results = pd.read_csv("/kaggle/working/runs/segment/train/results.csv")

# Meilleur epoch basé sur le mAP50 des masques
best_row = results.loc[results["metrics/mAP50(M)"].idxmax()]

# Résumé de cette exécution
run_metrics = pd.DataFrame([{
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "sample_size": len(train_files),
    "val_size": len(val_files),
    "epochs": len(results),
    "best_box_map50": float(best_row["metrics/mAP50(B)"]),
    "best_box_map50_95": float(best_row["metrics/mAP50-95(B)"]),
    "best_mask_map50": float(best_row["metrics/mAP50(M)"]),
    "best_mask_map50_95": float(best_row["metrics/mAP50-95(M)"]),
    "promoted": new_map > old_map
}])

# Fichier cumulatif
history_path = Path("/kaggle/working/all_runs_metrics.csv")

if history_path.exists():
    history = pd.read_csv(history_path)
    history = pd.concat([history, run_metrics], ignore_index=True)
else:
    history = run_metrics

history.to_csv(history_path, index=False)

print("Historique sauvegardé.")
print(history.tail())

if new_map <= old_map:
    raise SystemExit()

In [ ]:
# Créer un lien de téléchargement.
FileLink("/kaggle/working/best.pt")